# Validação do Grafo Disjuntivo e Cálculo do Cmax

Objetivo: validar `src/core/evaluator.py` (construção do grafo
disjuntivo e cálculo de `Cmax` via caminho crítico), conferindo
contra makespans calculados manualmente — Seção 2 da Especificação
Técnica.

Depende de `src/core/instance.py` e `src/core/evaluator.py` (Fase 3).

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))


In [ ]:
import numpy as np
import networkx as nx

from src.core.instance import Instancia
from src.core.evaluator import construir_grafo_disjuntivo, calcular_cmax


## Teste 1 — Permutação simples (2 jobs, 2 máquinas)

Instância pequena com tempos conhecidos, mesma ordem `[job0, job1]`
em ambas as máquinas (caso permutation flow shop). Makespan calculado
manualmente:

```
p[job0] = [3, 2]
p[job1] = [2, 3]

Máquina 0: job0 termina em 3; job1 começa em max(0, 3)=3, termina em 5
Máquina 1: job0 começa em max(3, 0)=3, termina em 5
           job1 começa em max(5, 5)=5, termina em 8

Cmax esperado = 8
```

In [ ]:
tempos_teste_1 = np.array([
    [3, 2],   # job 0
    [2, 3],   # job 1
])
instancia_teste_1 = Instancia(numero_jobs=2, numero_maquinas=2,
                               tempos_processamento=tempos_teste_1, nome="teste_1")

ordens_teste_1 = [
    np.array([0, 1]),   # máquina 0: job0 antes de job1
    np.array([0, 1]),   # máquina 1: mesma ordem (permutation flow shop)
]

cmax_obtido_1 = calcular_cmax(instancia_teste_1, ordens_teste_1)
print("Cmax obtido: ", cmax_obtido_1)
print("Cmax esperado:", 8)

assert cmax_obtido_1 == 8, "Cmax não bate com o cálculo manual!"
print("\nOK")


## Teste 2 — Non-permutation (ordens diferentes por máquina)

Mesma instância do Teste 1, mas com a ordem invertida na máquina 1
(`[job1, job0]`) — cenário que só é válido em NPFS, não em permutation
flow shop. Makespan calculado manualmente:

```
Máquina 0: job0 termina em 3; job1 começa em 3, termina em 5  (igual ao Teste 1)
Máquina 1 (ordem job1, job0):
    job1 começa em max(5, 0)=5, termina em 8
    job0 começa em max(3, 8)=8, termina em 10

Cmax esperado = 10
```

Note que o Cmax piorou (8 -> 10) por causa da ordem diferente na
máquina 1 — evidencia que o grafo está respeitando corretamente tanto
a precedência (dentro do job) quanto a disjunção (ordem na máquina).

In [ ]:
ordens_teste_2 = [
    np.array([0, 1]),   # máquina 0: igual ao Teste 1
    np.array([1, 0]),   # máquina 1: ordem invertida
]

cmax_obtido_2 = calcular_cmax(instancia_teste_1, ordens_teste_2)
print("Cmax obtido: ", cmax_obtido_2)
print("Cmax esperado:", 10)

assert cmax_obtido_2 == 10, "Cmax não bate com o cálculo manual!"
print("\nOK")


## Teste 3 — Verificação estrutural do grafo (DAG, sem ciclos)

Confere que o grafo construído é sempre um DAG válido (propriedade
garantida por construção, Seção 1: `m` permutações independentes
nunca geram ciclo) e que tem o número esperado de vértices e arestas.

In [ ]:
grafo_teste = construir_grafo_disjuntivo(instancia_teste_1, ordens_teste_2)

numero_vertices_esperado = instancia_teste_1.numero_jobs * instancia_teste_1.numero_maquinas
print("Vértices:", grafo_teste.number_of_nodes(), "| esperado:", numero_vertices_esperado)
print("Arestas: ", grafo_teste.number_of_edges())
print("É DAG (acíclico)?", nx.is_directed_acyclic_graph(grafo_teste))

assert grafo_teste.number_of_nodes() == numero_vertices_esperado
assert nx.is_directed_acyclic_graph(grafo_teste), "Grafo disjuntivo não deveria ter ciclos!"
print("\nOK")


## Teste 4 — Consistência contra simulação independente (instância aleatória)

Implementa aqui, de forma independente (sem usar o `evaluator.py`),
uma simulação direta das fórmulas `Start`/`Finish` da Seção 2, e
compara o resultado com `calcular_cmax` para várias instâncias e
ordens aleatórias. Serve como checagem cruzada da implementação real.

In [ ]:
def calcular_cmax_simulacao_manual(instancia, ordens_por_maquina):
    """Implementação independente das fórmulas da Seção 2, para comparação."""
    n, m = instancia.numero_jobs, instancia.numero_maquinas
    p = instancia.tempos_processamento

    finish = np.zeros((n, m))

    # posição de cada job na fila de cada máquina
    posicao_na_maquina = np.zeros((m, n), dtype=int)
    for k in range(m):
        for posicao, job in enumerate(ordens_por_maquina[k]):
            posicao_na_maquina[k, job] = posicao

    # processa em ordem topológica simples: por posição crescente em cada máquina,
    # respeitando que Start depende de Finish(job, k-1) e Finish(job_anterior, k)
    for k in range(m):
        for job in ordens_por_maquina[k]:
            finish_precedencia = finish[job, k - 1] if k > 0 else 0.0

            posicao = posicao_na_maquina[k, job]
            if posicao > 0:
                job_anterior = ordens_por_maquina[k][posicao - 1]
                finish_fila = finish[job_anterior, k]
            else:
                finish_fila = 0.0

            start = max(finish_precedencia, finish_fila)
            finish[job, k] = start + p[job, k]

    return finish.max()


gerador_aleatorio = np.random.default_rng(123)
numero_testes_aleatorios = 20

for indice_teste in range(numero_testes_aleatorios):
    n = gerador_aleatorio.integers(2, 8)
    m = gerador_aleatorio.integers(2, 5)
    tempos = gerador_aleatorio.integers(1, 50, size=(n, m))
    instancia_aleatoria = Instancia(n, m, tempos, nome=f"aleatoria_{indice_teste}")

    ordens_aleatorias = [gerador_aleatorio.permutation(n) for _ in range(m)]

    cmax_evaluator = calcular_cmax(instancia_aleatoria, ordens_aleatorias)
    cmax_manual = calcular_cmax_simulacao_manual(instancia_aleatoria, ordens_aleatorias)

    assert cmax_evaluator == cmax_manual, (
        f"Divergência no teste {indice_teste}: evaluator={cmax_evaluator}, manual={cmax_manual}"
    )

print(f"OK — {numero_testes_aleatorios} instâncias aleatórias, evaluator.py bate com a simulação manual.")


## Conclusão

O grafo disjuntivo e o cálculo de `Cmax` (`src/core/evaluator.py`)
estão corretos: batem com o cálculo manual em casos conhecidos
(permutation e non-permutation), o grafo é sempre um DAG válido, e o
resultado é consistente com uma simulação independente das fórmulas
da Seção 2 em instâncias e ordens aleatórias.